<a href="https://colab.research.google.com/github/dcolognesi/SEO/blob/Descrizioni/python_script_per_analisi_sito_finale_da_integrare_con_l'agente_seo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib.parse import urlparse, urljoin, parse_qs
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import logging
import re
from bs4 import BeautifulSoup
import hashlib
import warnings
import xml.etree.ElementTree as ET
from datetime import datetime
import networkx as nx
import operator
import json # Import json for pretty printing results

# Configure basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configure warnings to be captured by logging
logging.captureWarnings(True)

class WarningCatcher(logging.Handler):
    """A logging handler that captures warnings."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.warnings = []

    def emit(self, record):
        self.warnings.append(record.getMessage())

# Add the warning catcher to the root logger
warning_handler = WarningCatcher()
logging.getLogger("py.warnings").addHandler(warning_handler)


class CrawlerUtils:
    @staticmethod
    def create_session(retries=3, backoff_factor=0.3, status_forcelist=(429, 500, 502, 503, 504)):
        """Crea una sessione requests con retry."""
        session = requests.Session()
        retry = Retry(
            total=retries,
            read=retries,
            connect=retries,
            backoff_factor=backoff_factor,
            status_forcelist=list(status_forcelist),
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount('http://', adapter)
        session.mount('https://', adapter)
        return session

    @staticmethod
    def normalize_url(url):
        """Normalizza l'URL rimuovendo frammenti e potenzialmente parametri di tracciamento."""
        parsed = urlparse(url)
        # Mantiene i parametri di query, rimuove solo il frammento
        return parsed._replace(fragment="").geturl()

    @staticmethod
    def get_domain(url):
        """Estrae il dominio base dall'URL."""
        return urlparse(url).netloc

    @staticmethod
    def is_internal(url, base_domain):
        """Verifica se un URL è interno al dominio base."""
        return urlparse(url).netloc == base_domain

    @staticmethod
    def is_html(response):
        """Verifica se la risposta è di tipo HTML."""
        content_type = response.headers.get('Content-Type', '')
        return 'html' in content_type.lower()

class RobotsTxtAnalyzer:
    def __init__(self, session, base_url):
        self.session = session
        self.base_url = base_url
        self.robots_url = urljoin(base_url, "/robots.txt")
        self.disallowed_paths = []
        self.sitemap_urls = []
        self._fetch_and_parse()

    def _fetch_and_parse(self):
        """Scarica e analizza il file robots.txt."""
        try:
            response = self.session.get(self.robots_url, timeout=5)
            response.raise_for_status() # Solleva un errore per codici di stato non 2xx
            self._parse_content(response.text)
            logger.info(f"Robots.txt scaricato e analizzato da {self.robots_url}")
        except requests.RequestException as e:
            logger.warning(f"Impossibile scaricare o analizzare robots.txt da {self.robots_url}: {e}")

    def _parse_content(self, content):
        """Parsa il contenuto del file robots.txt."""
        user_agent = "*"  # Consideriamo le regole per tutti gli user-agent
        current_rules = []
        in_our_user_agent_section = False

        for line in content.splitlines():
            line = line.strip()
            if not line or line.startswith('#'):
                continue

            parts = line.split(':', 1)
            if len(parts) < 2:
                continue

            directive = parts[0].strip().lower()
            value = parts[1].strip()

            if directive == "user-agent":
                if value == user_agent or value == "*":
                    in_our_user_agent_section = True
                    current_rules = [] # Reset rules for this user-agent
                else:
                    in_our_user_agent_section = False
            elif in_our_user_agent_section:
                if directive == "disallow":
                    current_rules.append(value)
                elif directive == "sitemap":
                    self.sitemap_urls.append(value)

        self.disallowed_paths = current_rules

    def allows_crawl(self, url):
        """Verifica se un URL è permesso dal robots.txt."""
        path = urlparse(url).path
        for disallowed in self.disallowed_paths:
            # Semplice controllo basato su startswith per ora
            if path.startswith(disallowed):
                return False
        return True

    def get_sitemap_urls(self):
        """Restituisce l'elenco degli URL delle sitemap."""
        return self.sitemap_urls

    def analyze(self):
        """Restituisce i risultati dell'analisi robots.txt."""
        return {
            "robots_url": self.robots_url,
            "disallowed_paths": self.disallowed_paths,
            "sitemap_urls": self.sitemap_urls
        }

class SitemapValidator:
    def __init__(self, session):
        self.session = session
        self.processed_sitemaps = set()

    def fetch_and_extract_urls(self, sitemap_urls):
        """Scarica e estrae URL da una lista di sitemap URL."""
        all_urls = set()
        errors = []
        to_process = deque(sitemap_urls)

        while to_process:
            sitemap_url = to_process.popleft()

            if sitemap_url in self.processed_sitemaps:
                continue

            logger.info(f"Processing sitemap: {sitemap_url}")
            self.processed_sitemaps.add(sitemap_url)

            try:
                response = self.session.get(sitemap_url, timeout=10)
                response.raise_for_status()
                content = response.content

                if 'sitemapindex' in content.lower().decode('utf-8', errors='ignore'):
                    # È un sitemap index, estrai URL di sitemap figli
                    root = ET.fromstring(content)
                    for child in root.findall('.//{http://www.sitemaps.org/schemas/sitemap/0.9}sitemap'):
                        loc = child.find('{http://www.sitemaps.org/schemas/sitemap/0.9}loc')
                        if loc is not None and loc.text:
                            to_process.append(loc.text)
                            logger.debug(f"Found child sitemap: {loc.text}")
                elif 'urlset' in content.lower().decode('utf-8', errors='ignore'):
                    # È una sitemap normale, estrai URL di pagina
                    root = ET.fromstring(content)
                    for url_elem in root.findall('.//{http://www.sitemaps.org/schemas/sitemap/0.9}url'):
                        loc = url_elem.find('{http://www.sitemaps.org/schemas/sitemap/0.9}loc')
                        if loc is not None and loc.text:
                            all_urls.add(CrawlerUtils.normalize_url(loc.text))
                            # logger.debug(f"Found page URL: {loc.text}")
                else:
                     logger.warning(f"Unknown sitemap format for {sitemap_url}")


            except requests.RequestException as e:
                logger.error(f"Error fetching sitemap {sitemap_url}: {e}")
                errors.append({"url": sitemap_url, "error": str(e)})
            except ET.ParseError as e:
                logger.error(f"Error parsing sitemap XML {sitemap_url}: {e}")
                errors.append({"url": sitemap_url, "error": f"XML parse error: {e}"})
            except Exception as e:
                logger.error(f"Unexpected error processing sitemap {sitemap_url}: {e}")
                errors.append({"url": sitemap_url, "error": f"Unexpected error: {e}"})


        return list(all_urls), errors, list(self.processed_sitemaps)

class PageAnalyzer:
    def __init__(self, url, response):
        self.url = url
        self.response = response
        self.soup = None # Inizializza soup a None

    def _get_soup(self):
        """Crea e restituisce l'oggetto BeautifulSoup."""
        if self.soup is None:
            try:
                # Usa response.content per evitare problemi di encoding
                self.soup = BeautifulSoup(self.response.content, 'html.parser')
            except Exception as e:
                logger.error(f"Error parsing HTML for {self.url}: {e}")
                self.soup = None # Assicurati che soup rimanga None in caso di errore
        return self.soup

    def _is_noindex(self):
        """Verifica se la pagina ha un meta noindex o header X-Robots-Tag."""
        # Check X-Robots-Tag header
        x_robots = self.response.headers.get('X-Robots-Tag', '').lower()
        if 'noindex' in x_robots:
            return True

        # Check meta robots tag in HTML
        soup = self._get_soup()
        if soup:
            meta_robots = soup.find('meta', attrs={'name': 'robots'})
            if meta_robots and 'content' in meta_robots.attrs:
                content = meta_robots['content'].lower()
                if 'noindex' in content:
                    return True
        return False

    def _get_canonical_url(self):
        """Estrae l'URL canonico dalla pagina."""
        soup = self._get_soup()
        if soup:
            canonical_link = soup.find('link', rel='canonical', href=True)
            if canonical_link and canonical_link['href']:
                # Risolvi URL relativo rispetto all'URL della pagina corrente
                return urljoin(self.url, canonical_link['href'])
        return None

    def _get_content_hash(self):
        """Calcola un hash del contenuto principale della pagina."""
        soup = self._get_soup()
        if soup:
            try:
                # Tentativo di hashare il body o l'intero HTML se il body non c'è
                content_to_hash = str(soup.find('body') if soup.find('body') else soup)
                # Rimuovi spazi bianchi multipli e caratteri speciali che non influenzano il contenuto visuale
                content_to_hash = re.sub(r'\s+', '', content_to_hash)
                content_to_hash = content_to_hash.encode('utf-8')
                return hashlib.md5(content_to_hash).hexdigest()
            except Exception as e:
                logger.warning(f"Could not calculate content hash for {self.url}: {e}")
                return None
        return None

    def _get_url_parameters(self):
        """Estrae i parametri di query dall'URL."""
        return parse_qs(urlparse(self.url).query)

    def _get_images_missing_alt(self):
        """Trova immagini senza attributo alt."""
        soup = self._get_soup()
        if soup:
            # Trova tutte le immagini <img> e filtra quelle senza alt o con alt vuoto
            missing_alt = [
                img['src'] for img in soup.find_all('img')
                if 'alt' not in img.attrs or not img['alt'].strip()
            ]
            return missing_alt
        return []

    def _extract_internal_links(self, base_url, domain):
        """Estrae i link interni validi dalla pagina."""
        links = []
        soup = self._get_soup()
        if not soup:
            return links # Ritorna lista vuota se non c'è soup

        parsed_base = urlparse(base_url)

        for link in soup.find_all('a', href=True):
            href = link['href']
            if not href or href.startswith('#') or href.startswith('mailto:') or href.startswith('tel:'):
                continue

            # Risolvi URL relativo rispetto all'URL della pagina corrente
            new_url = urljoin(self.url, href)
            parsed_new_url = urlparse(new_url)

            # Pulisci solo il frammento (#), mantieni la query
            new_url_clean = parsed_new_url._replace(fragment="").geturl()

            if parsed_new_url.netloc == domain:
                links.append(new_url_clean)
        return links


    def analyze(self):
        """Analizza la pagina per vari aspetti SEO e estrae i link."""
        if not CrawlerUtils.is_html(self.response):
            return None # Non analizzare pagine non HTML

        soup = self._get_soup()
        if not soup:
            # Se non è stato possibile creare la soup, non possiamo analizzare
            return {"url": self.url, "error": "Could not parse HTML", "internal_links": []}


        noindex = self._is_noindex()
        canonical_url = self._get_canonical_url()
        content_hash = self._get_content_hash()
        parameters = self._get_url_parameters()
        missing_alt_images = self._get_images_missing_alt()

        domain = urlparse(self.url).netloc
        # base_url = f"{urlparse(self.url).scheme}://{domain}" # Non necessario, urljoin usa l'URL corrente
        internal_links = self._extract_internal_links(self.url, domain) # Usa self.url come base

        analysis_data = {
            "url": self.url,
            "status_code": self.response.status_code,
            "response_time_seconds": self.response.elapsed.total_seconds(),
            "is_noindex": noindex,
            "canonical_url": canonical_url,
            "content_hash": content_hash,
            "url_parameters": parameters,
            "is_redirect": self.response.is_redirect,
            "redirect_location": self.response.headers.get('Location'),
            "images_missing_alt": missing_alt_images,
            "internal_links": internal_links # Aggiungiamo i link scoperti
        }
        return analysis_data


class SEOCrawlabilityAnalyzer:
    def __init__(self, start_url, config=None):
        self.start_url = CrawlerUtils.normalize_url(start_url)
        self.base_url = f"{urlparse(self.start_url).scheme}://{urlparse(self.start_url).netloc}"
        self.domain = CrawlerUtils.get_domain(self.start_url)

        self.config = {
            "max_pages": 100,
            "max_depth": 5,
            "max_workers": 5,
            "crawl_delay": 0.1,
            "retries": 3,
            "backoff_factor": 0.3
        }
        if config:
            self.config.update(config)

        self.session = CrawlerUtils.create_session(
            retries=self.config["retries"],
            backoff_factor=self.config["backoff_factor"]
        )

        self.robots_analyzer = RobotsTxtAnalyzer(self.session, self.base_url)
        self.sitemap_validator = SitemapValidator(self.session)

        self.crawled_pages_data = {} # {url: page_analysis_data}
        self.response_errors = {"crawl_errors": [], "processing_errors": []}
        self.redirects_followed = {} # {from_url: to_url}
        self.discovered_urls = set()
        self.queue = deque()

        # Attributes for internal linking analysis
        self.link_graph = None
        self.inlinks_count = None
        self.outlinks_count = None
        self.orphan_pages = None
        self.high_inlinks_pages = None
        self.pagerank = {} # Initialize as empty dictionary
        self.betweenness_centrality = {} # Initialize as empty dictionary


        # Aggregation results
        self.aggregated_results = {
            "total_pages_crawled": 0,
            "unique_pages": 0, # Based on canonical/content hash
            "noindex_pages": [],
            "redirects": {},
            "duplicate_content": {}, # Based on content hash
            "pages_with_missing_alt": [],
            "crawl_errors": [],
            "sitemap_errors": [],
            "warnings": [],
            "processed_sitemaps": [],
            "robots_txt_info": {},
            # Add internal linking metrics here later in _aggregate_results
            "internal_linking_metrics": {},
            "summary": ""
        }

    def _can_crawl(self, url):
        """Verifica se l'URL può essere scansionato in base a robots.txt e se è interno."""
        if not CrawlerUtils.is_internal(url, self.domain):
            logger.debug(f"Skipping external URL: {url}")
            return False
        if not self.robots_analyzer.allows_crawl(url):
            logger.debug(f"Skipping disallowed by robots.txt: {url}")
            return False
        return True

    def _follow_redirects(self, initial_url):
        """Segui le catene di redirect e registra il percorso."""
        history = self.crawled_pages_data.get(initial_url, {}).get('redirect_history', [])
        if not history:
             # Se non c'è history nella page_data (es. primo redirect), usa response history
            if initial_url in self.crawled_pages_data and hasattr(self.crawled_pages_data[initial_url].get('response'), 'history'):
                 history = [h.url for h in self.crawled_pages_data[initial_url]['response'].history] + [initial_url]
            else:
                 # Fallback se non c'è history or response object
                 logger.debug(f"No redirect history available for {initial_url}")
                 return

        for i in range(len(history) - 1):
            self.redirects_followed[history[i]] = history[i+1]
            logger.debug(f"Redirect recorded: {history[i]} -> {history[i+1]}")


    def _process_page(self, url):
        """Elabora una singola pagina: la scansiona e analizza."""
        try:
            time.sleep(self.config["crawl_delay"])
            logger.debug(f"Fetching {url}")
            response = self.session.get(url, timeout=10)

            # Store response object temporarily to access history if needed
            page_data = {
                 "url": response.url, # Use the final URL after redirects
                 "status_code": response.status_code,
                 "response_time_seconds": response.elapsed.total_seconds(),
                 "is_redirect": response.is_redirect,
                 "redirect_location": response.headers.get('Location'),
                 "redirect_history": [h.url for h in response.history] + [response.url], # Capture history
                 "internal_links": [], # Default a lista vuota
                 "response": response # Store response temporarily
             }


            if CrawlerUtils.is_html(response):
                page_analyzer = PageAnalyzer(response.url, response) # Use final URL after redirects
                analysis_data = page_analyzer.analyze() # Ora questo contiene anche 'internal_links'
                if analysis_data:
                     page_data.update(analysis_data)
                else:
                     # Handle case where HTML parsing failed
                     page_data["error"] = "HTML parsing failed"

            # Remove temporary response object before returning
            if "response" in page_data:
                 del page_data["response"]

            return page_data

        except requests.RequestException as e:
            logger.error(f"Impossibile scansionare {url}: {e}")
            # Return a dictionary with basic info and error, including empty internal_links
            return {"url": url, "error": str(e), "internal_links": [], "status_code": None, "response_time_seconds": None, "is_redirect": False, "redirect_location": None, "redirect_history": []}
        except Exception as e:
             logger.error(f"Unexpected error in _process_page for {url}: {e}")
             # Return a dictionary with basic info and error, including empty internal_links
             return {"url": url, "error": f"Unexpected error: {e}", "internal_links": [], "status_code": None, "response_time_seconds": None, "is_redirect": False, "redirect_location": None, "redirect_history": []}


    def _analyze_internal_linking(self):
        """Costruisce il grafo dei link interni e calcola le metriche."""
        logger.info("Avvio analisi del linking interno...")
        self.link_graph = {}
        self.inlinks_count = {}
        self.outlinks_count = {}
        self.orphan_pages = []
        self.high_inlinks_pages = []
        self.pagerank = {}
        self.betweenness_centrality = {}


        # Build the link graph
        for source_url, page_data in self.crawled_pages_data.items():
            # Ensure we only process pages where internal_links were extracted successfully
            if 'internal_links' in page_data:
                if source_url not in self.link_graph:
                     # Initialize the list for the source_url if it doesn't exist
                     # This ensures pages with 0 outlinks are still keys in the graph dict
                     self.link_graph[source_url] = []

                # Add outgoing links
                for target_url in page_data['internal_links']:
                    self.link_graph[source_url].append(target_url)


        # Add any crawled page that might not have outgoing links to the graph structure
        # This ensures all crawled pages are considered as potential nodes in the graph
        for crawled_url in self.crawled_pages_data.keys():
            if crawled_url not in self.link_graph:
                self.link_graph[crawled_url] = [] # Page was crawled but has no outgoing links


        logger.info(f"Grafo dei link costruito con {len(self.link_graph)} nodi (pagine scansionate considerate).")


        # Calculate inlinks and outlinks count
        # Re-calculate based on the finalized link_graph to include all crawled pages
        self.inlinks_count = {url: 0 for url in self.link_graph.keys()} # Initialize all crawled pages with 0 inlinks
        self.outlinks_count = {}


        for source_url, target_urls in self.link_graph.items():
             self.outlinks_count[source_url] = len(target_urls)
             for target_url in target_urls:
                  # Only count inlinks for pages that are in our crawled set (internal links)
                  if target_url in self.inlinks_count: # Check if the target is a crawled page
                      self.inlinks_count[target_url] += 1
                  # Optionally, you could count external inlinks too, but for internal linking analysis,
                  # focusing on internal targets is more relevant.

        logger.info(f"Calcolati inlinks e outlinks per {len(self.link_graph)} pagine.")


        # Identify orphan pages: crawled pages with 0 inlinks from other crawled pages
        # Now that inlinks_count is based on all crawled pages, this is more accurate
        self.orphan_pages = [
            url for url, count in self.inlinks_count.items() if count == 0
        ]


        logger.info(f"Trovate {len(self.orphan_pages)} pagine orfane (0 inlinks all'interno del set scansionato).")


        # Identify pages with high inlinks
        if self.inlinks_count:
            sorted_inlinks = sorted(self.inlinks_count.items(), key=operator.itemgetter(1), reverse=True)
            top_n = min(10, len(sorted_inlinks)) # Top 10 or fewer if less than 10
            self.high_inlinks_pages = sorted_inlinks[:top_n]
            logger.info(f"Identificate {len(self.high_inlinks_pages)} pagine con il maggior numero di inlinks.")
        else:
             self.high_inlinks_pages = []
             logger.info("Nessuna page con inlinks trovata.")


        # Build NetworkX graph for centrality metrics
        G = nx.DiGraph()
        if self.link_graph:
            for source_url, target_urls in self.link_graph.items():
                for target_url in target_urls:
                    # Only add edges between crawled pages
                    if target_url in self.crawled_pages_data:
                        G.add_edge(source_url, target_url)

            # Ensure all crawled pages are nodes in the graph, even if they have no links
            for url in self.crawled_pages_data.keys():
                 if url not in G.nodes():
                      G.add_node(url)


            logger.info(f"Grafo NetworkX creato con {G.number_of_nodes()} nodi e {G.number_of_edges()} archi.")


            # Calculate PageRank
            try:
                # Handle disconnected components by calculating PageRank on each component
                # and combining the results. Or use personalization.
                # A simpler approach is to calculate on the whole graph and accept that
                # disconnected nodes will have a base PageRank value (alpha).
                # Ensure the graph is not empty before calculating PageRank
                if G.number_of_nodes() > 0:
                    self.pagerank = nx.pagerank(G, alpha=0.85)
                    logger.info("Calcolato PageRank.")
                else:
                     logger.warning("Grafo NetworkX vuoto. Impossibile calcolare PageRank.")
                     self.pagerank = {}

            except nx.NetworkXError as e:
                logger.warning(f"Impossibile calcolare PageRank: {e}")
                self.pagerank = {} # Ensure it's empty if calculation fails
            except Exception as e:
                 logger.warning(f"Errore inatteso durante il calcolo del PageRank: {e}")
                 self.pagerank = {}


            # Calculate Betweenness Centrality (can be slow for large graphs)
            if G.number_of_nodes() > 0 and G.number_of_nodes() < 500: # Avoid calculation on huge graphs
                try:
                    self.betweenness_centrality = nx.betweenness_centrality(G)
                    logger.info("Calcolato Betweenness Centrality.")
                except nx.NetworkXError as e:
                    logger.warning(f"Impossibile calcolare Betweenness Centrality: {e}")
                    self.betweenness_centrality = {} # Ensure it's empty if calculation fails
                except Exception as e:
                    logger.warning(f"Errore inatteso durante il calcolo della Betweenness Centrality: {e}")
                    self.betweenness_centrality = {}
            else:
                if G.number_of_nodes() >= 500:
                    logger.info("Betweenness Centrality calculation skipped for large graph (>= 500 nodes).")
                else:
                    logger.warning("Grafo NetworkX vuoto. Impossibile calcolare Betweenness Centrality.")
                self.betweenness_centrality = {} # Ensure it's empty if skipped

        else:
             logger.warning("Grafo dei link vuoto. Impossibile costruire grafo NetworkX o calcolare metriche di centralità.")
             self.pagerank = {}
             self.betweenness_centrality = {}


        logger.info("Analisi linking interno completata.")



    def run_full_analysis(self):
        """Esegue l'intero flusso di analisi con un ciclo di crawling corretto."""
        logger.info(f"Avvio analisi completa per il dominio: {self.domain}")
        start_time = time.time()
        warning_handler.warnings.clear() # Clear warnings from previous runs

        # 1. Analisi Robots.txt
        self.aggregated_results["robots_txt_info"] = self.robots_analyzer.analyze()
        initial_sitemap_urls = self.robots_analyzer.get_sitemap_urls()
        logger.info(f"Trovate {len(initial_sitemap_urls)} URL di sitemap in robots.txt.")


        # 2. Analisi Sitemap (basata su URL trovati in robots.txt)
        sitemap_page_urls, sitemap_errors, processed_sitemaps = self.sitemap_validator.fetch_and_extract_urls(initial_sitemap_urls)
        self.aggregated_results["sitemap_errors"] = sitemap_errors
        self.aggregated_results["processed_sitemaps"] = processed_sitemaps
        logger.info(f"Trovate {len(sitemap_page_urls)} URL nelle sitemaps.")

        # 3. Crawling
        self.queue = deque([(self.start_url, 0)]) # Start with the initial URL
        self.discovered_urls = {self.start_url}
        self.crawled_pages_data = {} # Reset data for this run


        # Aggiungiamo gli URL della sitemap alla coda se non superiamo il limite
        for url in sitemap_page_urls:
            normalized_url = CrawlerUtils.normalize_url(url)
            if normalized_url not in self.discovered_urls and len(self.discovered_urls) < self.config["max_pages"]:
                self.discovered_urls.add(normalized_url)
                self.queue.append((normalized_url, 1)) # Consider URLs from sitemap at depth 1

        logger.info(f"Inizio crawling con {len(self.queue)} URL iniziali nella coda (inclusi sitemap).")

        # Dizionario per tenere traccia dei futures e dei loro URL/profondità
        futures = {}

        with ThreadPoolExecutor(max_workers=self.config["max_workers"]) as executor:
            # Loop principale: sottomette task finché ci sono URL nella coda o task attivi
             while self.queue or futures:
                # Sottomette nuovi task finché il pool non è pieno o la coda è vuota
                while self.queue and len(futures) < self.config["max_workers"] * 2 and len(self.crawled_pages_data) < self.config["max_pages"]:
                    url, depth = self.queue.popleft()

                    # Normalizza l'URL prima di controllare lo stato
                    normalized_url = CrawlerUtils.normalize_url(url)

                    if normalized_url in self.crawled_pages_data:
                        logger.debug(f"Skipping already crawled: {normalized_url}")
                        continue
                    if depth > self.config["max_depth"]:
                        logger.debug(f"Skipping max depth reached: {normalized_url}")
                        continue
                    if not self._can_crawl(normalized_url):
                        logger.debug(f"Skipping disallowed: {normalized_url}")
                        continue

                    logger.debug(f"Submitting task for: {normalized_url} at depth {depth}")
                    future = executor.submit(self._process_page, normalized_url)
                    futures[future] = (normalized_url, depth)
                    self.discovered_urls.add(normalized_url) # Marca come scoperto appena sottomesso


                # Processa i risultati dei task completati
                # Usa list(futures) per poter rimuovere elementi dal dizionario durante l'iterazione
                for future in as_completed(list(futures)):
                     if len(self.crawled_pages_data) >= self.config["max_pages"]:
                         # Cancella i task rimanenti se il limite è stato raggiunto
                         for remaining_future in futures:
                             remaining_future.cancel()
                         break # Esci dal ciclo as_completed

                     url, depth = futures.pop(future) # Rimuovi il task completato dal dizionario

                     try:
                        page_data = future.result()

                        if page_data:
                             # Use the final URL from page_data as the key if available, else use the original URL
                             final_url = page_data.get("url", url)
                             # Only add valid page data (not just error placeholders) if the crawl was attempted
                             # and the final URL is within the domain if redirects happened.
                             # A simple check: if status_code is not None, it was likely a valid attempt.
                             if page_data.get("status_code") is not None or "error" not in page_data:
                                self.crawled_pages_data[final_url] = page_data
                                logger.debug(f"Processed: {final_url} (from {url})")
                             else:
                                 # Store error data separately if it couldn't be processed
                                 self.response_errors["crawl_errors"].append(page_data)
                                 logger.error(f"Crawl error for {url}: {page_data.get('error', 'Unknown error')}")

                        else:
                             # This case should ideally not happen if _process_page returns data even on error
                             logger.error(f"Received empty page_data for {url}")


                     except Exception as e:
                         logger.error(f"Errore durante l'elaborazione del risultato per {url}: {e}")
                         self.response_errors["processing_errors"].append({"url": url, "error": str(e)})

                # Add a small delay between processing batches to avoid overwhelming
                if futures: # Apply delay only if there are still active (or just completed) tasks
                     time.sleep(0.1)


        logger.info(f"Scansione completa. Analizzate {len(self.crawled_pages_data)} pagine valide.")


        # 4. Analisi del Linking Interno
        # Ensure crawled_pages_data is populated before analyzing links
        if self.crawled_pages_data:
            self._analyze_internal_linking()
        else:
            logger.warning("Nessuna pagina scansionata con successo. Salto analisi linking interno.")
            # Initialize linking attributes to empty states if crawling failed
            self.link_graph = {}
            self.inlinks_count = {}
            self.outlinks_count = {}
            self.orphan_pages = []
            self.high_inlinks_pages = []
            self.pagerank = {}
            self.betweenness_centrality = {}



        # 5. Post-analisi e aggregazione dei risultati
        self._aggregate_results()

        end_time = time.time()
        duration = end_time - start_time
        logger.info(f"Analisi completa terminata in {duration:.2f} secondi.")


        # 6. Costruzione del dizionario finale dei risultati
        final_results = {
            "config": self.config,
            "start_url": self.start_url,
            "domain": self.domain,
            "crawled_pages_data": self.crawled_pages_data,
            "response_errors": self.response_errors,
            "redirects_followed": self.redirects_followed,
            "aggregated_results": self.aggregated_results, # Include all aggregated data
            "warnings": warning_handler.warnings,
            "duration_seconds": duration
        }

        final_results["summary"] = self._generate_summary(final_results)
        return final_results


    def _aggregate_results(self):
        """Aggrega i dati raccolti durante il crawling e l'analisi del linking interno."""
        self.aggregated_results["total_pages_crawled"] = len(self.crawled_pages_data)
        self.aggregated_results["crawl_errors"] = self.response_errors["crawl_errors"]

        content_hashes = {}
        canonical_urls = {}
        noindex_pages = []
        pages_with_missing_alt = []
        redirects = {}

        for url, data in self.crawled_pages_data.items():
            # Aggregate redirects
            if data.get('is_redirect'):
                 redirects[url] = data.get('redirect_location', 'N/A')
                 # If redirect history is available, record the chain
                 history = data.get('redirect_history', [])
                 for i in range(len(history) - 1):
                      self.redirects_followed[history[i]] = history[i+1]


            # Aggregate noindex pages
            if data.get('is_noindex'):
                noindex_pages.append(url)

            # Aggregate pages with missing alt text
            if data.get('images_missing_alt'):
                pages_with_missing_alt.append({"url": url, "missing_alt_images": data["images_missing_alt"]})

            # Aggregate content hashes for duplicate content detection
            content_hash = data.get('content_hash')
            if content_hash:
                if content_hash not in content_hashes:
                    content_hashes[content_hash] = []
                content_hashes[content_hash].append(url)

            # Aggregate canonical URLs for unique page count and canonicalization issues
            canonical_url = data.get('canonical_url')
            if canonical_url:
                canonical_normalized = CrawlerUtils.normalize_url(canonical_url)
                if canonical_normalized not in canonical_urls:
                    canonical_urls[canonical_normalized] = []
                canonical_urls[canonical_normalized].append(url)
            else:
                 # Pages without canonical tags are also counted
                 if url not in canonical_urls:
                     canonical_urls[url] = [] # Use URL itself if no canonical tag


        # Detect duplicate content (more than one URL sharing the same content hash)
        duplicate_content = {
            ch: urls for ch, urls in content_hashes.items() if len(urls) > 1
        }

        # Count unique pages based on normalized canonical URL or the URL itself if no canonical
        unique_pages_count = len(canonical_urls)


        self.aggregated_results["redirects"] = redirects
        self.aggregated_results["noindex_pages"] = noindex_pages
        self.aggregated_results["pages_with_missing_alt"] = pages_with_missing_alt
        self.aggregated_results["duplicate_content"] = duplicate_content
        self.aggregated_results["unique_pages"] = unique_pages_count
        self.aggregated_results["canonical_urls_map"] = canonical_urls # Store for potential further analysis
        self.aggregated_results["warnings"] = warning_handler.warnings # Capture collected warnings


        # Add internal linking metrics to aggregated_results
        self.aggregated_results["internal_linking_metrics"] = {
            "total_pages_with_outlinks": len(self.outlinks_count) if self.outlinks_count is not None else 0,
            "total_unique_pages_receiving_inlinks": len(self.inlinks_count) if self.inlinks_count is not None else 0,
            "orphan_pages_count": len(self.orphan_pages) if self.orphan_pages is not None else 0,
            "orphan_pages_list": self.orphan_pages if self.orphan_pages is not None else [],
            "pages_with_highest_inlinks": self.high_inlinks_pages if self.high_inlinks_pages is not None else [],
            "pagerank_scores": self.pagerank if self.pagerank is not None else {},
            "betweenness_centrality_scores": self.betweenness_centrality if self.betweenness_centrality is not None else {}
        }


    def _generate_summary(self, results):
        """Genera un riepilogo testuale dei risultati."""
        summary = "Riepilogo Analisi SEO Crawlability:\n"
        summary += f"Dominio analizzato: {results['domain']}\n"
        summary += f"URL di partenza: {results['start_url']}\n"
        summary += f"Durata totale: {results['duration_seconds']:.2f} secondi\n"
        summary += f"Pagine scansionate: {results['aggregated_results']['total_pages_crawled']}\n"
        summary += f"Pagine uniche (basate su canonical/hash): {results['aggregated_results']['unique_pages']}\n"
        summary += f"Pagine noindexed: {len(results['aggregated_results']['noindex_pages'] if results['aggregated_results']['noindex_pages'] is not None else [])}\n"
        summary += f"Redirects trovati: {len(results['aggregated_results']['redirects'] if results['aggregated_results']['redirects'] is not None else {})}\n"
        summary += f"Pagine con contenuto duplicato (basato su hash): {len(results['aggregated_results']['duplicate_content'] if results['aggregated_results']['duplicate_content'] is not None else {})}\n"
        summary += f"Pagine con immagini senza ALT: {len(results['aggregated_results']['pages_with_missing_alt'] if results['aggregated_results']['pages_with_missing_alt'] is not None else [])}\n"
        summary += f"Errori di crawling: {len(results['response_errors']['crawl_errors'] if results['response_errors']['crawl_errors'] is not None else [])}\n"
        summary += f"Errori Sitemap: {len(results['aggregated_results']['sitemap_errors'] if results['aggregated_results']['sitemap_errors'] is not None else [])}\n"
        summary += f"Warnings catturati: {len(results['warnings'] if results['warnings'] is not None else [])}\n"

        # Add internal linking summary
        internal_linking_metrics = results['aggregated_results'].get('internal_linking_metrics', {})
        summary += "\nAnalisi Linking Interno:\n"
        summary += f"Pagine analizzate per inlinks/outlinks: {internal_linking_metrics.get('total_pages_with_outlinks', 'N/A')}\n"
        summary += f"Pagine uniche che ricevono inlinks: {internal_linking_metrics.get('total_unique_pages_receiving_inlinks', 'N/A')}\n"
        summary += f"Pagine orfane trovate: {internal_linking_metrics.get('orphan_pages_count', 'N/A')}\n"
        summary += f"Pagine con alto numero di inlinks: {len(internal_linking_metrics.get('pages_with_highest_inlinks', []))}\n"
        summary += f"PageRank calcolato: {'Sì' if internal_linking_metrics.get('pagerank_scores') else 'No'}\n"
        summary += f"Betweenness Centrality calcolata: {'Sì' if internal_linking_metrics.get('betweenness_centrality_scores') else 'No'}\n"


        # Aggiungi dettagli se ci sono errori o problemi
        if results['aggregated_results']['noindex_pages']:
            summary += f"\nPagine noindexed ({len(results['aggregated_results']['noindex_pages'])}): {', '.join(results['aggregated_results']['noindex_pages'][:5])}{'...' if len(results['aggregated_results']['noindex_pages']) > 5 else ''}\n"
        if results['aggregated_results']['duplicate_content']:
             # Check if duplicate_content is not empty before trying to popitem
             if results['aggregated_results']['duplicate_content']:
                # Safely get the first group of duplicate URLs
                first_duplicate_group = next(iter(results['aggregated_results']['duplicate_content'].values()), [])
                summary += f"\nGruppi di contenuto duplicato ({len(results['aggregated_results']['duplicate_content'])}) (primo gruppo): {', '.join(first_duplicate_group[:5])}{'...' if len(first_duplicate_group) > 5 else ''}\n"
             else:
                 summary += "\nNessun gruppo di contenuto duplicato trovato.\n"
        if results['aggregated_results']['pages_with_missing_alt']:
             summary += f"\nPagine con immagini senza ALT ({len(results['aggregated_results']['pages_with_missing_alt'])}): {', '.join([p['url'] for p in results['aggregated_results']['pages_with_missing_alt']][:5])}{'...' if len(results['aggregated_results']['pages_with_missing_alt']) > 5 else ''}\n"
        if results['response_errors']['crawl_errors']:
            summary += f"\nErrori di crawling ({len(results['response_errors']['crawl_errors'])}): {', '.join([e['url'] for e in results['response_errors']['crawl_errors']][:5])}{'...' if len(results['response_errors']['crawl_errors']) > 5 else ''}\n"
        if results['aggregated_results']['sitemap_errors']:
             summary += f"\nErrori Sitemap ({len(results['aggregated_results']['sitemap_errors'])}): {', '.join([e['url'] for e in results['aggregated_results']['sitemap_errors']][:5])}{'...' if len(results['aggregated_results']['sitemap_errors']) > 5 else ''}\n"
        if internal_linking_metrics.get('orphan_pages_list'):
             summary += f"\nPagine orfane ({internal_linking_metrics['orphan_pages_count']}): {', '.join(internal_linking_metrics['orphan_pages_list'][:5])}{'...' if internal_linking_metrics['orphan_pages_count'] > 5 else ''}\n"
        if internal_linking_metrics.get('pages_with_highest_inlinks'):
             summary += f"\nPagine con più inlinks ({len(internal_linking_metrics['pages_with_highest_inlinks'])}): {', '.join([f"{url} ({count})" for url, count in internal_linking_metrics['pages_with_highest_inlinks'][:5]])}{'...' if len(internal_linking_metrics['pages_with_highest_inlinks']) > 5 else ''}\n"


        return summary


# Example Usage (can be uncommented and run separately)
# if __name__ == "__main__":
#     # URL del sito web da analizzare
#     start_url = "https://www.villaggiodellamadre.org"  # SOSTITUISCI CON L'URL REALE DEL SITO

#     # Parametri di configurazione (opzionali)
#     config = {
#         "max_pages": 100,  # Numero massimo di pagine da scansionare
#         "max_depth": 5,    # Profondità massima di scansione
#         "max_workers": 5,  # Numero di thread concorrenti
#         "crawl_delay": 0.1 # Ritardo tra le richieste in secondi
#     }

#     # Crea un'istanza dell'analizzatore
#     analyzer = SEOCrawlabilityAnalyzer(start_url, config)

#     # Esegui l'analisi completa
#     results = analyzer.run_full_analysis()

#     # Stampa o elabora i risultati (es. in formato JSON)
#     print(json.dumps(results, indent=4))

#     # Puoi anche accedere a risultati specifici come:
#     # print("\n--- Summary ---")
#     # print(results["summary"])
#     # print("\n--- Orphan Pages ---")
#     # print(results["aggregated_results"]["internal_linking_metrics"]["orphan_pages_list"])